# Projecting Aggression network into new mice

In [ ]:
import numpy as np
import sklearn.decomposition as dp
import pickle
import sys,os
import numpy.random as rand
from sklearn.linear_model import LogisticRegression as LR
from sklearn.metrics import auc,roc_curve,roc_auc_score
from sklearn.utils.random import sample_without_replacement
from scipy import io
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

## Load in the labels and backprojection

In [ ]:
labelDict = io.loadmat('Windows.mat')
mouse = np.squeeze(labelDict['mouse'])
group = np.squeeze(labelDict['group'])
time = np.squeeze(labelDict['time'])
condition = np.squeeze(labelDict['condition'])
behavior = np.squeeze(labelDict['behavior'])

In [ ]:
sys.path.append('/home/austin/DataAnalysis')
from data_tools import load_data

In [ ]:
power,coherence,granger,labels = load_data('/media/austin/ThickBoy__1/DataAgression/CL_baseline_all_validate2.mat',
                                          fBounds=(1,56),feature_list=['power','coherence','granger'])

power = 10*power
power = power.astype(np.float32)
power[power>6] = 6 

coherence = coherence.astype(np.float32)
granger = np.exp(granger)
granger[granger>10] = 10
granger = granger.astype(np.float32)

X = np.hstack((power,coherence,granger))


In [ ]:
Ex = np.mean(X,axis=0)
np.mean((X-Ex)**2)

In [ ]:
np.mean(X**2)

In [ ]:
model_dict = pickle.load(open('/home/austin/Aggression/Experiments/Joint/Trial2_reweighted_true.p','rb'))
A_enc = model_dict['A']
B_enc = model_dict['B']
components_ = model_dict['components_']
S_train_l = tf.nn.softplus(np.dot(X,A_enc) + B_enc)
S_train = S_train_l.numpy()
Xr = np.dot(S_train,components_)
print(np.mean((X-Xr)**2))

In [ ]:
model_dict.keys()

In [ ]:
print('>>>>>>>>>>>>>')
print(model_dict['recon_rand_tr'])
print(model_dict['recon_rand_te'])
print('>>>>>>>>>>>>>')

print(model_dict['recon_nmf_tr'])
print(model_dict['recon_nmf_te'])
print('>>>>>>>>>>>>>')

print(model_dict['recon_snmf_tr'])
print(model_dict['recon_snmf_te'])
print('>>>>>>>>>>>>>')


# (Condition=4,Behavior=1) vs (behavior=2,condition=4,6,8)

In [ ]:
y = np.zeros(len(mouse))
idx_pos = (condition==4)&(behavior==1)
y[idx_pos] = 1

In [ ]:
idx_neg = ((behavior==2)&((condition==4)|(condition==6)|(condition==8)))
idx_tot = idx_pos|idx_neg

In [ ]:
Scores_supervised_limited = Scores_supervised[idx_tot]
Scores_unsupervised_limited = Scores_unsupervised[idx_tot]
y_limited = y[idx_tot]
mouse_limited = mouse[idx_tot]
print(np.sum(idx_tot))

## Actually go through and compute the AUCs by mouse for the supervised factor

In [ ]:
mice = np.unique(mouse_limited)
nMice = len(mice)

In [ ]:
auc_sups = np.zeros(nMice)
for i in range(nMice):
    idx_mouse = mouse_limited==mice[i]
    y_true = y_limited[idx_mouse]
    y_hat = Scores_supervised_limited[idx_mouse]
    auc_sups[i] = roc_auc_score(y_true,y_hat)

In [ ]:
for i in range(nMice):
    print('%s AUC: %0.2f'%(mice[i],auc_sups[i]))
mean = np.mean(auc_sups)
ci = np.std(auc_sups)/np.sqrt(nMice)*1.96
print('Test set AUC confidence interval: %0.2f +- %0.3f'%(mean,ci))

## Now lets look at the unsupervised factors

In [ ]:
auc_unsup = np.zeros((nMice,7))
for i in range(nMice):
    for j in range(7):
        idx_mouse = mouse_limited==mice[i]
        y_true = y_limited[idx_mouse]
        y_hat = Scores_unsupervised_limited[idx_mouse,j]
        auc_unsup[i,j] = roc_auc_score(y_true,y_hat)
means = np.mean(auc_unsup,axis=0)
stds = np.std(auc_unsup,axis=0)
ci = 1.96*stds/np.sqrt(nMice)

In [ ]:
for j in range(7):
    mainStr = 'Test set AUC CI Factor %d: %0.2f +- %0.2f'%(j+1,means[j],ci[j])
    print(mainStr)

## Finally get the correlation between the factors

In [ ]:
S_tot = np.zeros((len(mouse_limited),8))
S_tot[:,0] = Scores_supervised_limited
S_tot[:,1:] = Scores_unsupervised_limited
cc = np.corrcoef(S_tot.T)

## Correlation between the most positive factor and supervised factor

In [ ]:
print('Correlation is',cc[0,1])

In [ ]:
d = {'Mouse':mice,'Supervised AUC':auc_sups,'Unsupervised 1':auc_unsup[:,0],
    'Unsupervised 2':auc_unsup[:,1],'Unsupervised 3':auc_unsup[:,2],
    'Unsupervised 4':auc_unsup[:,3],'Unsupervised 5':auc_unsup[:,4],
    'Unsupervised 6':auc_unsup[:,5],'Unsupervised 7':auc_unsup[:,6],}
df = pd.DataFrame(data=d)

In [ ]:
df.to_csv("SingleFactor_Condition2.csv",index=False,sep=',')

# (Condition=4,Behavior=1) vs (behavior=0,condition=4,6,8)

In [ ]:
idx_neg = ((behavior==0)&((condition==4)|(condition==6)|(condition==8)))
idx_tot = idx_pos|idx_neg

Scores_supervised_limited = Scores_supervised[idx_tot]
Scores_unsupervised_limited = Scores_unsupervised[idx_tot]
y_limited = y[idx_tot]
mouse_limited = mouse[idx_tot]

In [ ]:
mice = np.unique(mouse_limited)
nMice = len(mice)

auc_sups = np.zeros(nMice)
for i in range(nMice):
    idx_mouse = mouse_limited==mice[i]
    y_true = y_limited[idx_mouse]
    y_hat = Scores_supervised_limited[idx_mouse]
    auc_sups[i] = roc_auc_score(y_true,y_hat)

### Supervised factor performance

In [ ]:
for i in range(nMice):
    print('%s AUC: %0.2f'%(mice[i],auc_sups[i]))
mean = np.mean(auc_sups)
ci = np.std(auc_sups)/np.sqrt(nMice)*1.96
print('Test set AUC confidence interval: %0.2f +- %0.3f'%(mean,ci))

### Unsupervised factors performance

In [ ]:
auc_unsup = np.zeros((nMice,7))
for i in range(nMice):
    for j in range(7):
        idx_mouse = mouse_limited==mice[i]
        y_true = y_limited[idx_mouse]
        y_hat = Scores_unsupervised_limited[idx_mouse,j]
        auc_unsup[i,j] = roc_auc_score(y_true,y_hat)
means = np.mean(auc_unsup,axis=0)
stds = np.std(auc_unsup,axis=0)
ci = 1.96*stds/np.sqrt(nMice)

for j in range(7):
    mainStr = 'Test set AUC CI Factor %d: %0.2f +- %0.2f'%(j+1,means[j],ci[j])
    print(mainStr)

### Correlation between the factors

In [ ]:
print('Correlation is',cc[0,7])

In [ ]:
d = {'Mouse':mice,'Supervised AUC':auc_sups,'Unsupervised 1':auc_unsup[:,0],
    'Unsupervised 2':auc_unsup[:,1],'Unsupervised 3':auc_unsup[:,2],
    'Unsupervised 4':auc_unsup[:,3],'Unsupervised 5':auc_unsup[:,4],
    'Unsupervised 6':auc_unsup[:,5],'Unsupervised 7':auc_unsup[:,6],}
df = pd.DataFrame(data=d)

In [ ]:
df.to_csv("SingleFactor_Condition0.csv",index=False,sep=',')